# Biology problems

Three mathematically explicit problems in cell-therapy assay interpretation. All values are synthetic. The code exposes units, model assumptions, boundary behavior and independent checks.

## Setup

Run from the repository root after installing `requirements.lock`. The validation runner executes this notebook and retains a separate executed copy. All code blocks below run against the same versioned implementation.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
import analysis
OUT = Path("artifacts/run")

## 1. Finite ligand and receptor binding

Problem: receptor total = 100 nM, ligand total = 10 nM, Kd = 1 nM. Find bound complex and receptor occupancy. Derive (R−C)(L−C)=Kd·C from mass balance and one-to-one equilibrium. Solve the quadratic with a cancellation-resistant expression, then check against a bracketed numerical root. C cannot exceed min(R,L). The common L/(Kd+L) expression requires free ligand, so substituting total ligand is invalid when depletion matters.

## 2. Counting beads and uncertainty

Problem: 120 target events, 800 bead events, 10,000 beads added and 100 µL sample. The concentration is (120/800)×(10,000/100) = 15 cells/µL. Condition on the observed target-plus-bead count to obtain an exact binomial Clopper-Pearson interval, then transform its limits to a Poisson-rate ratio. The interval excludes uncertainty in pipetting, mixing and viability. A zero target count gives an estimate of zero with a positive upper bound; zero observed beads makes the concentration undefined.

## 3. Rare-cell detection with background

Without background, the chance of at least one true detection is 1−(1−p·s)^N. Solve for the smallest integer N achieving the requested confidence, and verify N and N−1. Then add a per-cell false-positive rate: an “any positive” rule can produce frequent false alarms. Calibrate a count threshold using the binomial blank distribution and calculate power separately. A sample-size calculation alone is not an experimentally validated limit of detection.

## Execute the complete analysis

Inspect the returned results and tables. No outcome is hand-written into the code or edited to create a better story.

In [ ]:
summary = analysis.run(OUT)
print(json.dumps(summary, indent=2))

In [ ]:
print(analysis.complex_concentration(100, 10, 1))
print(analysis.occupancy(100, 10, 1))
display(Image(filename=str(OUT / "binding.png")))

In [ ]:
print(analysis.bead_count(120, 800, 10_000, 100))
print(analysis.bead_count(0, 800, 10_000, 100))

In [ ]:
display(pd.read_csv(OUT / "rare_event_power.csv"))
display(Image(filename=str(OUT / "rare_event_power.png")))

## What was validated

`tests/` checks hand-calculated examples, independent formulations, malformed inputs and relevant invariants. `python scripts/validate.py` runs pytest, the full pipeline and this notebook. Passing these checks is bounded computational evidence, not clinical validation. See `reports/reference/FINDINGS.md` for actual results and limitations.